In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))
from scripts.market_api import data_pull
from scripts.data_construction import data_construction

print("Initializing historical data ingestion pipeline...\n")

# ----------------- LAYER 1: Treatment Group (Booster Cookies) -----------------
url_cookie = "https://sky.coflnet.com/api/bazaar/BOOSTER_COOKIE/history"
output_cookie = "cookie_market_historical.csv"

data_pull(url_cookie, output_cookie)
path_cookie = "data/raw/cookie_market_historical.csv"
# Load and slice the last 200 rows natively
cleaned_cookie_data = data_construction(path_cookie).tail(200).reset_index(drop=True)


# ----------------- LAYER 2: Control Group (Perfect Ruby Gemstones) -----------------
url_gem = 'https://sky.coflnet.com/api/bazaar/PERFECT_AMBER_GEM/history'
output_gem = "gemstone_market_historical.csv"

data_pull(url_gem, output_gem)
path_gem = "data/raw/gemstone_market_historical.csv"
# Load and slice the last 200 rows natively
cleaned_gemstone_data = data_construction(path_gem).tail(200).reset_index(drop=True)

print(f"\nConnection to scripts successful. Datasets manually trimmed to last {len(cleaned_cookie_data)} records.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

round(cleaned_cookie_data[['buy','sell', 'buyVolume', 'sellVolume']].agg(
    ['mean',
     'std',
     'max',
     'min']
),2)

round(cleaned_gemstone_data[['buy','sell', 'buyVolume', 'sellVolume']].agg(
    ['mean',
     'std',
     'max',
     'min']
),2)

In [ ]:
differenced_cookie_data = cleaned_cookie_data[['buy', 'sell','timestamp']].copy()
differenced_gemstone_data = cleaned_gemstone_data[['buy', 'sell','timestamp']].copy()

# Cookie
differenced_cookie_data['diff_sell'] = cleaned_cookie_data['sell'] - cleaned_cookie_data['sell'].shift(1)
differenced_cookie_data['diff_buy'] = cleaned_cookie_data['buy'] - cleaned_cookie_data['buy'].shift(1)

# Gemstone
differenced_gemstone_data['diff_sell'] = cleaned_gemstone_data['sell'] - cleaned_gemstone_data['sell'].shift(1)
differenced_gemstone_data['diff_buy'] = cleaned_gemstone_data['buy'] - cleaned_gemstone_data['buy'].shift(1)

# Standard summary statistics for cookies
summary = round(differenced_cookie_data[['diff_buy', 'diff_sell']].agg(['mean', 'std', 'min', 'max']), 4)
print(summary)

# Standard summary statistics for gemstones
summary = round(differenced_gemstone_data[['diff_buy', 'diff_sell']].agg(['mean', 'std', 'min', 'max']), 4)
print(summary)

# Calculate Cookie Z-scores
z_buy = (differenced_cookie_data['diff_buy'] - differenced_cookie_data['diff_buy'].mean()) / differenced_cookie_data['diff_buy'].std()
z_sell = (differenced_cookie_data['diff_sell'] - differenced_cookie_data['diff_sell'].mean()) / differenced_cookie_data['diff_sell'].std()

# Calculate Gemstone Z-scores
z_buy = (differenced_gemstone_data['diff_buy'] - differenced_gemstone_data['diff_buy'].mean()) / differenced_gemstone_data['diff_buy'].std()
z_sell = (differenced_gemstone_data['diff_sell'] - differenced_gemstone_data['diff_sell'].mean()) / differenced_gemstone_data['diff_sell'].std()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# LEFT PLOT: Raw Buy Price
ax1.plot(cleaned_cookie_data['timestamp'], cleaned_cookie_data['buy'], 
         label='Buy Price', color='royalblue', linewidth=1.5)

ax1.set_title('Booster Cookie Buy Values Over Time', fontsize=13, fontweight='bold')
ax1.set_xlabel('Time', fontsize=11)
ax1.set_ylabel('Buy Value', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend()

# RIGHT PLOT: 1st Differenced
ax2.plot(differenced_cookie_data['timestamp'], differenced_cookie_data['diff_buy'], 
         label='Buy Price (1st Diff)', color='royalblue', linewidth=1.5)

ax2.set_title('Booster Cookie Buy Values (1st Diff) Over Time', fontsize=13, fontweight='bold')
ax2.set_xlabel('Time', fontsize=11)
ax2.set_ylabel('$\Delta$ Buy Value', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# LEFT PLOT: Raw Buy Price
ax1.plot(cleaned_gemstone_data['timestamp'], cleaned_gemstone_data['buy'], 
         label='Buy Price', color='royalblue', linewidth=1.5)

ax1.set_title('Ruby Gemstone Buy Values Over Time', fontsize=13, fontweight='bold')
ax1.set_xlabel('Time', fontsize=11)
ax1.set_ylabel('Buy Value', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend()

# RIGHT PLOT: 1st Differenced
ax2.plot(differenced_gemstone_data['timestamp'], differenced_gemstone_data['diff_buy'], 
         label='Buy Price (1st Diff)', color='royalblue', linewidth=1.5)

ax2.set_title('Ruby Gemstone Buy Values (1st Diff) Over Time', fontsize=13, fontweight='bold')
ax2.set_xlabel('Time', fontsize=11)
ax2.set_ylabel('$\Delta$ Buy Value', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()

fig.autofmt_xdate()
plt.tight_layout()
plt.show()